In [7]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bazaar_sales.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS orders;")
cursor.execute("DROP TABLE IF EXISTS customers;")

cursor.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    join_month TEXT
);""")

cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    amount REAL,
    product TEXT
);""")

cursor.executemany("INSERT INTO customers VALUES (?, ?, ?);", [
    (101, "Aman", "2026-01"), (102, "Priya", "2026-01"), 
    (103, "Rahul", "2026-02"), (104, "Sneha", "2026-02")
])


cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?);", [
    (1, 101, "2026-01-15", 4500.0, "Laptop"),
    (2, 102, "2026-01-20", 350.0, "Mouse"),
    (3, 101, "2026-02-10", 1200.0, "Monitor"),
    (4, 103, "2026-02-18", 2200.0, "Laptop"),
    (5, 102, "2026-03-05", 150.0, "Keyboard"),
    (6, 101, "2026-03-22", 3100.0, "Desk")
])

conn.commit()
print("'bazaar_sales.db' created!")

'bazaar_sales.db' created!


In [8]:
sql_query = """
SELECT
    customer_id,
    SUM(amount) AS total_filtered_spend
FROM orders
WHERE amount >= 200
GROUP BY customer_id
HAVING total_filtered_spend > 2000;
"""

df_result = pd.read_sql_query(sql_query, conn)

print("=== Filtered Query Results ===")
print(df_result)

=== Filtered Query Results ===
   customer_id  total_filtered_spend
0          101                8800.0
1          103                2200.0


In [9]:
df_orders = pd.read_sql_query("SELECT * FROM orders", conn)

pandas_result = df_orders.groupby("customer_id")["amount"].sum().reset_index()

print(pandas_result)

   customer_id  amount
0          101  8800.0
1          102   500.0
2          103  2200.0


In [10]:
sql_query = """
SELECT customer_id, SUM(amount) AS total_spend
FROM orders
GROUP BY customer_id;
"""

sql_result = pd.read_sql_query(sql_query, conn)

print(sql_result)

   customer_id  total_spend
0          101       8800.0
1          102        500.0
2          103       2200.0


In [11]:
test_query = """
SELECT 
    c.customer_id,
    c.name,
    c.join_month,
    o.order_id,
    o.amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id;
"""
print(pd.read_sql_query(test_query, conn))

   customer_id   name join_month  order_id  amount
0          101   Aman    2026-01       3.0  1200.0
1          101   Aman    2026-01       6.0  3100.0
2          101   Aman    2026-01       1.0  4500.0
3          102  Priya    2026-01       5.0   150.0
4          102  Priya    2026-01       2.0   350.0
5          103  Rahul    2026-02       4.0  2200.0
6          104  Sneha    2026-02       NaN     NaN


In [12]:
cohort_query = """
SELECT 
    c.join_month AS cohort,
    COUNT(DISTINCT o.customer_id) AS active_buyers,
    SUM(o.amount) AS total_cohort_revenue
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.join_month;
"""

df_cohorts = pd.read_sql_query(cohort_query, conn)
print(df_cohorts)

    cohort  active_buyers  total_cohort_revenue
0  2026-01              2                9300.0
1  2026-02              1                2200.0


In [13]:
cte_rolling_query = """
WITH Monthly_Sales_CTE AS (
    SELECT 
        strftime('%m', order_date) AS order_month,
        SUM(amount) AS monthly_revenue
    FROM orders
    GROUP BY strftime('%m', order_date)
)
SELECT 
    order_month,
    monthly_revenue,
    AVG(monthly_revenue) OVER (
        ORDER BY order_month
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_3_month_avg
FROM Monthly_Sales_CTE;
"""

df_rolling = pd.read_sql_query(cte_rolling_query, conn)
print(df_rolling)
conn.close()

  order_month  monthly_revenue  rolling_3_month_avg
0          01           4850.0          4850.000000
1          02           3400.0          4125.000000
2          03           3250.0          3833.333333
